In [1]:
import numpy as np
import pandas as pd

from sklearn.svm import SVR
from sklearn.linear_model import LinearRegression,Lasso, Ridge
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.compose import ColumnTransformer

In [2]:
df = pd.read_csv("feature_selection_data_outlier.csv")
pd.set_option('display.max_columns', None)
df.head()

,price,carpet_area,facing,bedrooms,bathrooms,balcony,furnish_type,prop_type,city,Store Room,ageing,location_area,luxury_category,floor_category
0,6.90,1092.0,North-East,3.0,3.0,0.0,Semifurnished,Flat,Mumbai,0,Relatively New,Parel,High,High Floor
1,4.50,1051.0,North-East,3.0,3.0,0.0,Unfurnished,Flat,Mumbai,0,Relatively Old,Ghatkopar,Medium,Mid Floor
2,7.30,1081.0,North-East,3.0,3.0,2.0,Semifurnished,Flat,Mumbai,1,Relatively Old,Goregaon,High,Low Floor
3,1.56,486.0,East,2.0,2.0,0.0,Unfurnished,Flat,Mumbai,0,New,Kandivali,Low,Mid Floor
4,1.43,625.0,East,2.0,2.0,0.0,Unfurnished,Flat,Mumbai,0,Old,Mulund,Medium,Low Floor


In [3]:
df.shape

(8862, 14)

In [4]:
x= df.drop(columns=['price'])
y=df['price']

In [5]:
y_transformed = np.log1p(y)

### Ordinal Encoder

In [6]:
enc_cols = ['facing','furnish_type','prop_type','city','ageing','location_area','floor_category','luxury_category']

In [ ]:
preprocessor=ColumnTransformer(
    [('num', StandardScaler(),['carpet_area','bedrooms','bathrooms','balcony','Store Room']),
     ('cat', OrdinalEncoder(handle_unknown="use_encoded_value",unknown_value=-1),enc_cols)],
    remainder='passthrough'
)

In [ ]:
pipeline = Pipeline(
    [('preprocessor',preprocessor),
     ('regressor',LinearRegression())]
)

In [ ]:
# K-fold cross-validation
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, x, y_transformed, cv=kfold, scoring='r2')

In [ ]:
scores.mean(),scores.std()

(np.float64(0.7370639775335099), np.float64(0.05899663061152207))

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(x,y_transformed,test_size=0.2,random_state=42)

In [ ]:
pipeline.fit(x_train,y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('num', StandardScaler(),
                                                  ['carpet_area', 'bedrooms',
                                                   'bathrooms', 'balcony',
                                                   'Store Room']),
                                                 ('cat',
                                                  OrdinalEncoder(handle_unknown='use_encoded_value',
                                                                 unknown_value=-1),
                                                  ['facing', 'furnish_type',
                                                   'prop_type', 'city',
                                                   'ageing', 'location_area',
                                                   'floor_category',
                                                   'luxury_category'])])),
                ('regressor', LinearRegression())])

In [ ]:
y_pred = pipeline.predict(x_test)
mean_absolute_error(y_test,y_pred)

0.19853232224702494

In [ ]:
y_pred=np.expm1(y_pred)
mean_absolute_error(np.expm1(y_test),y_pred)

0.8222471918045212

In [ ]:
def scoring(model_name, model, preprocessor):

    output = []

    output.append(model_name)

    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', model)
    ])

    # K-fold cross-validation
    kfold = KFold(n_splits=10, shuffle=True, random_state=42)
    scores = cross_val_score(pipeline, x, y_transformed, cv=kfold, scoring='r2')

    output.append(scores.mean())

    X_train, X_test, y_train, y_test = train_test_split(x,y_transformed,test_size=0.2,random_state=42)

    pipeline.fit(X_train,y_train)

    y_pred = pipeline.predict(X_test)

    y_pred = np.expm1(y_pred)

    output.append(mean_absolute_error(np.expm1(y_test),y_pred))

    return output


In [7]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor,ExtraTreesRegressor,GradientBoostingRegressor,AdaBoostRegressor
from sklearn.neural_network import MLPRegressor
import xgboost

In [ ]:
model_dict = {
    'linear_reg':LinearRegression(),
    'svr':SVR(),
    'ridge':Ridge(),
    'LASSO':Lasso(),
    'decision tree': DecisionTreeRegressor(),
    'random forest':RandomForestRegressor(),
    'extra trees': ExtraTreesRegressor(),
    'gradient boosting': GradientBoostingRegressor(),
    'adaboost': AdaBoostRegressor(),
    'mlp': MLPRegressor(),
    'xgboost':xgboost.XGBRegressor()
}

In [ ]:
model_op = []
for model_name, model in model_dict.items():
    model_op.append(scoring(model_name,model,preprocessor))

In [ ]:
model_op

[['linear_reg', np.float64(0.7370639775335099), 0.8222471918045212],
 ['svr', np.float64(0.7812026533608935), 0.7627913417783143],
 ['ridge', np.float64(0.7370842345581203), 0.8222829497841145],
 ['LASSO', np.float64(-0.0009298169514451837), 1.6160202740856904],
 ['decision tree', np.float64(0.7564434473712591), 0.731916354540633],
 ['random forest', np.float64(0.8715979990119331), 0.5372796135379094],
 ['extra trees', np.float64(0.8608083968377782), 0.54840538129934],
 ['gradient boosting', np.float64(0.8601156836156644), 0.5944082410821686],
 ['adaboost', np.float64(0.7167562513801158), 0.8587765911743734],
 ['mlp', np.float64(0.7988263793173142), 0.723945677935862],
 ['xgboost', np.float64(0.8851421696396324), 0.5026148767657422]]

In [ ]:
model_df = pd.DataFrame(model_op,columns=['name','r2','mae'])
model_df.sort_values('mae')

,name,r2,mae
10,xgboost,0.885142,0.502615
5,random forest,0.871598,0.537280
6,extra trees,0.860808,0.548405
7,gradient boosting,0.860116,0.594408
9,mlp,0.798826,0.723946
4,decision tree,0.756443,0.731916
1,svr,0.781203,0.762791
0,linear_reg,0.737064,0.822247
2,ridge,0.737084,0.822283
8,adaboost,0.716756,0.858777


### One Hot Encoder

In [ ]:
preprocessor=ColumnTransformer(
    [('num', StandardScaler(),['carpet_area','bedrooms','bathrooms','balcony','Store Room']),
     ('cat', OrdinalEncoder(handle_unknown="use_encoded_value",unknown_value=-1),enc_cols),
     ('cat1', OneHotEncoder(drop='first',handle_unknown='ignore'),['city','ageing','furnish_type'])],
    remainder='passthrough'
)

In [ ]:
pipeline = Pipeline(
    [('preprocessor',preprocessor),
     ('regressor',LinearRegression())]
)

In [ ]:
# K-fold cross-validation
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, x, y_transformed, cv=kfold, scoring='r2')

In [ ]:
scores.mean(),scores.std()

(np.float64(0.7393948363424991), np.float64(0.058798945740839215))

In [ ]:
pipeline.fit(x_train,y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('num', StandardScaler(),
                                                  ['carpet_area', 'bedrooms',
                                                   'bathrooms', 'balcony',
                                                   'Store Room']),
                                                 ('cat',
                                                  OrdinalEncoder(handle_unknown='use_encoded_value',
                                                                 unknown_value=-1),
                                                  ['facing', 'furnish_type',
                                                   'prop_type', 'city',
                                                   'ageing', 'location_area',
                                                   'floor_category',
                                                   'luxury_category']),
                                                 ('cat1',
                                                  OneHotEncoder(drop='first',
                                                                handle_unknown='ignore'),
                                                  ['city', 'ageing',
                                                   'furnish_type'])])),
                ('regressor', LinearRegression())])

In [ ]:
y_pred = pipeline.predict(x_test)
y_pred = np.expm1(y_pred)
mean_absolute_error(np.expm1(y_test),y_pred)

0.8154689093721271

In [ ]:
model_op = []
for model_name, model in model_dict.items():
    model_op.append(scoring(model_name,model,preprocessor))

In [ ]:
model_op

[['linear_reg', np.float64(0.7393948363424991), 0.8154689093721271],
 ['svr', np.float64(0.7824257245040984), 0.7532545600656604],
 ['ridge', np.float64(0.7394169189933758), 0.8154968058162722],
 ['LASSO', np.float64(-0.0009298169514451837), 1.6160202740856904],
 ['decision tree', np.float64(0.7599966755573888), 0.7284105843831011],
 ['random forest', np.float64(0.871014996395283), 0.5306707827913424],
 ['extra trees', np.float64(0.8593471099928323), 0.5529033471227316],
 ['gradient boosting', np.float64(0.859977112024304), 0.5906644531760643],
 ['adaboost', np.float64(0.711244058631772), 0.8501882765096658],
 ['mlp', np.float64(0.7899378870744955), 0.7258770651738349],
 ['xgboost', np.float64(0.8847409874780329), 0.4961353356331536]]

In [ ]:
model_df = pd.DataFrame(model_op,columns=['name','r2','mae'])
model_df.sort_values('mae')

,name,r2,mae
10,xgboost,0.884741,0.496135
5,random forest,0.871015,0.530671
6,extra trees,0.859347,0.552903
7,gradient boosting,0.859977,0.590664
9,mlp,0.789938,0.725877
4,decision tree,0.759997,0.728411
1,svr,0.782426,0.753255
0,linear_reg,0.739395,0.815469
2,ridge,0.739417,0.815497
8,adaboost,0.711244,0.850188


### OHE with PCA

In [ ]:
preprocessor=ColumnTransformer(
    [('num', StandardScaler(),['carpet_area','bedrooms','bathrooms','balcony','Store Room']),
     ('cat', OrdinalEncoder(handle_unknown="use_encoded_value",unknown_value=-1),enc_cols),
     ('cat1', OneHotEncoder(drop='first',handle_unknown='ignore',sparse_output=False),['city','ageing'])],
    remainder='passthrough'
)

In [ ]:
model_op = []
for model_name, model in model_dict.items():
    model_op.append(scoring(model_name,model,preprocessor))

In [ ]:
model_op

[['linear_reg', np.float64(0.7394001251468633), 0.815019693197744],
 ['svr', np.float64(0.7825837402569977), 0.7530493393303909],
 ['ridge', np.float64(0.7394221697296504), 0.8150442226468981],
 ['LASSO', np.float64(-0.0009298169514451837), 1.6160202740856904],
 ['decision tree', np.float64(0.7633775745962332), 0.7208559792060951],
 ['random forest', np.float64(0.8707524181639996), 0.5354549322911026],
 ['extra trees', np.float64(0.8602573020925076), 0.5456100619603182],
 ['gradient boosting', np.float64(0.8603873544486271), 0.5919820090036162],
 ['adaboost', np.float64(0.7127947549532538), 0.881487842685469],
 ['mlp', np.float64(0.7767664524130904), 0.7184330376268461],
 ['xgboost', np.float64(0.8864007562861854), 0.4950625408564727]]

In [ ]:
model_df = pd.DataFrame(model_op,columns=['name','r2','mae'])
model_df.sort_values('mae')

,name,r2,mae
10,xgboost,0.886401,0.495063
5,random forest,0.870752,0.535455
6,extra trees,0.860257,0.545610
7,gradient boosting,0.860387,0.591982
9,mlp,0.776766,0.718433
4,decision tree,0.763378,0.720856
1,svr,0.782584,0.753049
0,linear_reg,0.739400,0.815020
2,ridge,0.739422,0.815044
8,adaboost,0.712795,0.881488


### Target Encoder

In [ ]:
# %pip install category_encoders

Defaulting to user installation because normal site-packages is not writeable
  Obtaining dependency information for category_encoders from https://files.pythonhosted.org/packages/92/fb/908cb215a30b117bb079a767176038599a5447f2506e21aa2e90d0aabfff/category_encoders-2.8.1-py3-none-any.whl.metadata
  Using cached category_encoders-2.8.1-py3-none-any.whl.metadata (7.9 kB)
  Obtaining dependency information for scikit-learn>=1.6.0 from https://files.pythonhosted.org/packages/b2/3b/47b5eaee01ef2b5a80ba3f7f6ecf79587cb458690857d4777bfd77371c6f/scikit_learn-1.7.1-cp311-cp311-win_amd64.whl.metadata
  Obtaining dependency information for threadpoolctl>=3.1.0 from https://files.pythonhosted.org/packages/32/d5/f9a850d79b0851d1d4ef6456097579a9005b31fea68726a4ae5f2d82ddd9/threadpoolctl-3.6.0-py3-none-any.whl.metadata
Using cached category_encoders-2.8.1-py3-none-any.whl (85 kB)
   ---------------------------------------- 0.0/8.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.9 MB 

In [8]:
import category_encoders as ce
preprocessor=ColumnTransformer(
    [('num', StandardScaler(),['carpet_area','bedrooms','bathrooms','balcony','Store Room']),
     ('cat', OrdinalEncoder(handle_unknown="use_encoded_value",unknown_value=-1),enc_cols),
     ('cat1', OneHotEncoder(drop='first',handle_unknown='ignore',sparse_output=False),['city','ageing']),
     ('target_enc',ce.TargetEncoder(),['city'])],
    remainder='passthrough'
)

In [ ]:
model_output = []
for model_name,model in model_dict.items():
    model_output.append(scoring(model_name, model, preprocessor))

In [ ]:
model_output

[['linear_reg', np.float64(0.7394001251468633), 0.815019693197741],
 ['svr', np.float64(0.7825734565425139), 0.7531821454700175],
 ['ridge', np.float64(0.739422225185394), 0.8150435695419787],
 ['LASSO', np.float64(-0.0009298169514451837), 1.6160202740856904],
 ['decision tree', np.float64(0.7618136648151707), 0.7268512403897597],
 ['random forest', np.float64(0.8722693830415242), 0.5358082376939106],
 ['extra trees', np.float64(0.8598512120135101), 0.5485124079958444],
 ['gradient boosting', np.float64(0.8603345965016705), 0.5921028181645724],
 ['adaboost', np.float64(0.711809805539899), 0.8612103928331709],
 ['mlp', np.float64(0.7910260920250376), 0.707774894855929],
 ['xgboost', np.float64(0.8864007562861854), 0.4950625408564727]]

In [ ]:
model_df = pd.DataFrame(model_output,columns=['name','r2','mae'])
model_df.sort_values('mae')

,name,r2,mae
10,xgboost,0.886401,0.495063
5,random forest,0.872269,0.535808
6,extra trees,0.859851,0.548512
7,gradient boosting,0.860335,0.592103
9,mlp,0.791026,0.707775
4,decision tree,0.761814,0.726851
1,svr,0.782573,0.753182
0,linear_reg,0.739400,0.815020
2,ridge,0.739422,0.815044
8,adaboost,0.711810,0.861210


In [ ]:
preprocessor=ColumnTransformer(
    [('num', StandardScaler(),['carpet_area','bedrooms','bathrooms','balcony','Store Room']),
     ('cat', OrdinalEncoder(handle_unknown="use_encoded_value",unknown_value=-1),enc_cols),
     ('cat1', OneHotEncoder(drop='first',handle_unknown='ignore',sparse_output=False),['city','ageing']),
     ('target_enc',ce.TargetEncoder(),['location_area'])],
    remainder='passthrough'
)

In [ ]:
model_output = []
for model_name,model in model_dict.items():
    model_output.append(scoring(model_name, model, preprocessor))

In [ ]:
model_output

[['linear_reg', np.float64(0.8181102726117608), 0.6312741602959142],
 ['svr', np.float64(0.8359623486186883), 0.6503109789253911],
 ['ridge', np.float64(0.8181273121801), 0.6313478667026037],
 ['LASSO', np.float64(-0.0009298169514451837), 1.6160202740856904],
 ['decision tree', np.float64(0.7975212442773544), 0.6018910504327762],
 ['random forest', np.float64(0.8949849647982273), 0.45530957997246263],
 ['extra trees', np.float64(0.8950902334202396), 0.4534579485456218],
 ['gradient boosting', np.float64(0.8806150039253728), 0.5203782114284922],
 ['adaboost', np.float64(0.7532899524197663), 0.7651948792622354],
 ['mlp', np.float64(0.8440141697819218), 0.6038032402206749],
 ['xgboost', np.float64(0.8910700795061277), 0.46514961979467184]]

In [ ]:
model_df = pd.DataFrame(model_output,columns=['name','r2','mae'])
model_df.sort_values('mae')

,name,r2,mae
6,extra trees,0.895090,0.453458
5,random forest,0.894985,0.455310
10,xgboost,0.891070,0.465150
7,gradient boosting,0.880615,0.520378
4,decision tree,0.797521,0.601891
9,mlp,0.844014,0.603803
0,linear_reg,0.818110,0.631274
2,ridge,0.818127,0.631348
1,svr,0.835962,0.650311
8,adaboost,0.753290,0.765195
